# Day 6: Data Quality, Validation & Quarantine DLQ Engine

This notebook demonstrates running validation on intentional bad records and verifying routing to a quarantine Dead Letter Queue (DLQ).

In [ ]:
import sys
import os
# Add the src directory to the path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from src.quality.dq_circuit_breaker import run_dq_checks

spark = SparkSession.builder.appName("Day06_DQ_DLQ").getOrCreate()
print("Spark Session initialized.")

In [ ]:
# Create synthetic data with intentional bad records
schema = StructType([
    StructField("person_id", StringType(), True),
    StructField("heart_rate", IntegerType(), True)
])

data = [
    ("P001", 75),   # Valid
    (None, 80),     # Invalid (Null ID)
    ("P003", 999)   # Invalid (Out of bounds vitals)
]

df = spark.createDataFrame(data, schema)
print("Incoming Data Pipeline:")
df.show()

In [ ]:
# Run Data Quality Engine
valid_df, dlq_df = run_dq_checks(df)

print("Valid Data routed to Silver:")
valid_df.show()

print("Invalid Data routed to Quarantine DLQ:")
dlq_df.show(truncate=False)